# 03 Regenerate the Grad-CAM figure

**AI-Based Early Detection and Classification of Foot and Nail Conditions**

Produces `results/figures/mobilenetv2_gradcam.png`, the attention overlays for
Section 5.6.2 of the dissertation, and downloads it so it can be pasted into
the Word document.

Run the cells in order. Set **Runtime > Change runtime type > T4 GPU** first:
it works on CPU but scoring 1,248 test images takes several minutes there.

Nothing here retrains or re-tunes anything. It loads the model already saved on
Drive and scores the test split with it, which is the same deterministic pass
that produced the figures already in the dissertation, so the numbers it prints
should match Chapter 5. If they do not, stop and check which model was restored.


## 1. Mount Drive, and clone or update the repository


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO    = 'https://github.com/gurubasavarajharlapur-jpg/Dissertation_AI_FOOT_NAIL_DISEASE.git'
BRANCH  = 'claude/foot-nail-disease-ai-fyrdsb'
PROJECT = Path('/content/Dissertation_AI_FOOT_NAIL_DISEASE')
DRIVE   = Path('/content/drive/MyDrive/dissertation_foot_nail')

# Clone if absent, pull if present, so this cell is safe to re-run.
if (PROJECT / '.git').is_dir():
    !cd {PROJECT} && git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q --ff-only
else:
    !git clone -q --branch {BRANCH} {REPO} {PROJECT}

%cd {PROJECT}
!pip install -q -r requirements.txt
!git log --oneline -1


## 2. Restore the trained model and the processed images from Drive

The Grad-CAM figure needs two things: the trained MobileNetV2 in `models/`, and
the test images in `data/processed/`. Both are restored from the single archives
on Drive. Raw data is not needed and is not restored.


In [ ]:
!python src/colab_sync.py restore --what models processed
!python src/colab_sync.py status


In [ ]:
# Preflight: fail clearly here rather than with a stack trace further down.
import sys
sys.path.insert(0, '/content/Dissertation_AI_FOOT_NAIL_DISEASE')
from src import config

model_file = config.model_path('mobilenetv2')
test_index = config.PROCESSED_DIR / 'test.csv'   # where evaluate.py looks for it

print('trained model :', model_file, '->', 'found' if model_file.exists() else 'MISSING')
print('test split    :', test_index, '->', 'found' if test_index.exists() else 'MISSING')
print('processed dir :', config.PROCESSED_DIR, '->',
      f'{sum(1 for _ in config.PROCESSED_DIR.rglob("*.jpg"))} images'
      if config.PROCESSED_DIR.exists() else 'MISSING')

if not (model_file.exists() and test_index.exists()):
    raise SystemExit('Restore did not bring back what is needed. Check the archives on Drive '
                     'with: !python src/colab_sync.py status')


## 3. Check the GPU


In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__, '| Keras', tf.keras.__version__)
print('GPU:', [g.name for g in gpus] if gpus else
      'NONE. It will still work, just slowly. Runtime > Change runtime type > T4 GPU')


## 4. Run the evaluation, which writes the Grad-CAM figure

This prints the full test report as well. Compare the accuracy it reports with
98.32 percent in Chapter 5: they should be identical, because the model and the
split are the same. Inference latency may differ, since it depends on whichever
machine Colab allocated, and the dissertation quotes the earlier measurement.


In [ ]:
!python src/evaluate.py --model mobilenetv2


## 5. Look at the figure, then download it


In [ ]:
from IPython.display import Image, display

figure = config.FIGURES_DIR / 'mobilenetv2_gradcam.png'
print(figure, f'({figure.stat().st_size // 1024} KB)')
display(Image(str(figure), width=900))


In [ ]:
from google.colab import files
files.download(str(figure))


## 6. Save the regenerated results back to Drive

Optional, but it means the figure survives the runtime being recycled.


In [ ]:
!python src/colab_sync.py save --what results


## Where it goes in the dissertation

Section 5.6.2, Grad-CAM attention, immediately after the paragraph that begins
*Attention localises to the toes and nail plates*. Paste the image, then add a
caption below it in the same style as the others:

> **Figure 5.5.**  Grad-CAM overlays for MobileNetV2, including correct and
> misclassified predictions. Red and yellow mark the region that drove the
> prediction.

Figure 5.5 is free: the figures in Chapter 5 currently run 5.1 to 5.4. After
pasting it, select the whole document and press F9 so the List of Figures picks
it up, and add it there by hand if it does not, since a pasted image carries no
caption field.
